#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline
import evaluate

dataset = load_dataset("ag_news")
categories = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

dbu = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(dbu)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
tokenized_datasets = dataset.map(tokenize_function, batched=True)


model = AutoModelForSequenceClassification.from_pretrained(dbu, num_labels=4)

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

test_results = trainer.evaluate()
print(f"\n точность теста: {test_results['eval_accuracy']:.4f}")
trainer.save_model("./ag_news_model")
tokenizer.save_pretrained("./ag_news_model")


classifier = pipeline("text-classification", model="./ag_news_model", device=0 if training_args.device.type == "cuda" else -1)

examplenews = [
    "Tehran threatens Middle East's busiest port as Iran war enters its third week",
    "Rashad McCants: Kobe Bryant 'could've gotten at least 120, 130 points'",
    "Yaks may hint at a way to treat brain diseases like MS"
]
results = classifier(examplenews)
for text, res in zip(examplenews, results):
    label_id = int(res['label'].split('_')[-1])
    print(f"новость: {text[:50]}...")
    print(f"предсказанный класс: {categories[label_id]}, уверенность модели: {res['score']:.4f}\n")